# Road, Path, Way, Street

Languages tend to have many words for road, street, path, ... The goal of this notebook is to select several such words in Sumerian ([inti[way]N](https://oracc.museum.upenn.edu/epsd2/sux/o0031182), [kaskal[way]N](https://oracc.museum.upenn.edu/epsd2/sux/o0031651),  [harran[route]N](https://oracc.museum.upenn.edu/epsd2/sux/o0029974), and [esir[street]N](https://oracc.museum.upenn.edu/epsd2/sux/o0027271)) to see if we can differentiate between them computationally. The process takes four steps:

1. download and process raw data
2. establish collocation probabilities with PMI
3. create word embedding vectors with SVD
4. plot the results


# 0. Preparation
First `import` the necessary Python packages. If you work in the Computational Assyriology environment, that should work without problems.

In [20]:
import zipfile
import json
from tqdm.auto import tqdm
import os
import pickle
import requests
from collections import Counter
from itertools import combinations
from math import log
from pprint import pformat
from scipy.sparse import csc_matrix, diags
from scipy.sparse.linalg import svds
from sklearn import preprocessing
from nltk import ngrams
from numpy.linalg import norm
import ipywidgets as widgets
import pandas as pd
import gensim
import numpy as np
from ipywidgets import (
    HBox,
    VBox,
    fixed,
    interactive_output)
from IPython.display import display, clear_output
from sklearn.manifold import TSNE
from bokeh.io import output_notebook, show, output_file, save
from bokeh.plotting import figure, output_file, save
from bokeh.resources import INLINE
from bokeh.models import (
    ColumnDataSource,
    LabelSet,
    Legend,
    TapTool,
    OpenURL,
    HoverTool,
    Title
)
output_notebook()

Loading BokehJS ...

1. Download and Process

Create directories for the download files and for output. If these directories already exist, nothing happens.

In [2]:
DIRECTORIES = ['jsonzip', 'output']
for directory in DIRECTORIES: 
    os.makedirs(directory, exist_ok = True)

Which [ORACC](https://oracc.museum.upenn.edu) projects will be used?

In [3]:
projects = ["epsd2/literary", "obel", "dsst"]

The function for downloading the data from multiple projects with the `requests` module. If the file is not available the server will send back a status code 404 and the code will issue a warning. Some JSON files are very large. For that reason, the code divides the file into chunks of 1024 bytes each. Much of the code is there, essentially, to create a progress bar with the `tqdm` package.

If the project you wish to download is not available on the oracc.museum.upenn.edu server, you may also try the following addresses:

- https://build-oracc.museum.upenn.edu/json/
- http://oracc.ub.uni-muenchen.de/json/

In [4]:
def oracc_download(project_list):
    CHUNK = 1024

    for project in project_list:
        proj = project.replace("/", "-")
        url = f"https://oracc.museum.upenn.edu/json/{proj}.zip"
        localfile = f"jsonzip/{proj}.zip"

        try:
            with requests.get(url, stream=True, timeout=30) as r:

                r.raise_for_status() # catch any type of error in handling the request

                total_size = int(r.headers.get("content-length", 0))
                tqdm.write(f"Saving {url} → {localfile}")

                with (
                    open(localfile, "wb") as f,
                    tqdm(
                        total=total_size,
                        unit="B",
                        unit_scale=True,
                        desc=project,
                    ) as bar,
                ):
                    for chunk in r.iter_content(chunk_size=CHUNK):
                        if chunk:
                            f.write(chunk)
                            bar.update(len(chunk))

        except requests.exceptions.RequestException as e:
            tqdm.write(f"ERROR downloading {project}: {e}")

In [5]:
oracc_download(projects)

Saving https://oracc.museum.upenn.edu/json/epsd2-literary.zip → jsonzip/epsd2-literary.zip


epsd2/literary:   0%|          | 0.00/39.5M [00:00<?, ?B/s]

Saving https://oracc.museum.upenn.edu/json/obel.zip → jsonzip/obel.zip


obel:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Saving https://oracc.museum.upenn.edu/json/dsst.zip → jsonzip/dsst.zip


dsst:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

The ZIP file that you have downloaded contains the project data in JSON format (JavaScript Object Notation). 

JSON is a very explicit format and the ORACC JSON files are rather complex. The following code will pull from the JSON the lemmatizations of all the words in the format `lugal[king]N`. Each individual text will be represented by a list of such lemmatizations, where the lemmas are listed in the original order. 

The analysis that we will use looks at all the collocates in a `window` - for instance with a window size of 7 it will look at three words before and three words after the target word. For that reason it is important to capture text boundaries, breakage, etc. All of that is available in the JSON data.

The code takes care of the following special situations:

- Unlemmatized words - words that are damaged or unknown should not be included. They are replaced by an underscore ("_") as placeholder. Such words do not contribute to our analysis, but should also not be removed because we do not want to create artificial neighbors.
- Words that are not in Sumerian are removed (for instance Akkadian glosses).
- Breakage in the text

In [6]:
lemm_l = []
ids_ = []
BREAKAGE = {'illegible', 'traces', 'missing', 'effaced','other', 'blank', 'ruling'}

In [7]:
def parsejson(text, current_lemmas):
    for JSONobject in text["cdl"]:
        if "cdl" in JSONobject: 
            parsejson(JSONobject, current_lemmas)
        elif JSONobject.get("state", "") in BREAKAGE or JSONobject.get("subtype", "")[:5] in ["seal ", "envel"]:
            # at any logical or physical break, or where a seal or envelope starts
            if len(current_lemmas) > 1:
                lemm_l.append(current_lemmas.copy()) #append a copy of current_lemmas
            current_lemmas.clear()                   # and start over with an empty version of l
            continue
        elif JSONobject.get("ftype", "") == "yn": 
            continue                                     # skip yearnames
        elif "f" in JSONobject:          # copy all the lemmatization data in the variable word
            word = JSONobject["f"]
            if not word.get("lang", "").startswith("sux"): #only Sumerian and Emesal
                continue
            if word.get("pos", "") == "n":  # omit numbers
                continue
            if "cf" in word:
                #for some reason some words appear without pos. Provisionally treated as Noun
                lemma = f"{word['cf']}[{word['gw']}]{word.get('pos', 'N')}"  
                lemma = lemma.replace(' ', '-')
            else:
                lemma = "_" # if word is unlemmatized enter a place holder
            if "x" in word.get("cf","").lower():  # partly damaged PN; enter placeholder
                lemma = "_"
            current_lemmas.append(lemma)           # append the lemmatization to current_lemmas
    return

In order to run the `parsejson()` function we do not need to unzip the entire files that we downloaded. Instead, we use the `zipfile` module to provide the list of files inside the zip and to read the ones we need.

In [8]:
for project in projects:
    p = project.replace("/", "-")
    file = f"jsonzip/{p}.zip"
    try:
        z = zipfile.ZipFile(file)       # create a Zipfile object
    except:
        print(f"{file} does not exist or is not a valid ZIP file")
        continue
    
    files = z.namelist()     # list of all the files in the ZIP
    files = [name for name in files if "corpusjson" in name and name.endswith('.json')]                                                                                                  #that holds all the P, Q, and X numbers.

    for filename in tqdm(files, desc = project):      #iterate over the file names
        id_no = filename[-13:-5]
    
        if id_no in ids_ and not "X" in id_no: # Check if P/Q number is already in there
            continue        # a text may appear in multiple projects
        
        id_text = project + id_no # id_text is, for instance, blms/P414332
        ids_.append(id_text)

        current_lemmas = []

        try:
            text = z.read(filename).decode("utf-8")
            data_json = json.loads(text)

            parsejson(data_json, current_lemmas)

            if len(current_lemmas) > 1:
                lemm_l.append(current_lemmas.copy())

        except (
            KeyError,
            json.JSONDecodeError,
            UnicodeDecodeError,
            OSError,
        ) as e:
            print(f"{id_text} unavailable or incomplete: {e}")

        finally:
            current_lemmas.clear()

epsd2/literary:   0%|          | 0/1022 [00:00<?, ?it/s]

obel:   0%|          | 0/671 [00:00<?, ?it/s]

dsst:   0%|          | 0/535 [00:00<?, ?it/s]

The first two texts in our corpus

In [9]:
lemm_l[:2]

[['urbara[wolf]N', 'gu[neck]N', 'ak[do]V/t', 'urmah[lion]N', 'il[raise]V/t'],
 ['_',
  '_',
  'nam[fate]N',
  '_',
  '_',
  '_',
  '_',
  'su[flesh]N',
  'dar[split]V/t',
  '_',
  '_',
  '_',
  'a[arm]N',
  'aŋ[measure]V/t',
  '_',
  '_',
  '_',
  'gub[stand]V/i',
  '_',
  'zig[rise]V/i',
  'uru[city]N',
  'gul[destroy]V/t',
  '_',
  'bur[tear]V/t',
  'huluŋal[evil]AJ',
  '_',
  '_',
  'šita[weapon]N',
  '_',
  'uš[poison]N',
  'šum[give]V/t',
  '_',
  'mah[great]V/i',
  '_',
  'aŋal[strong]AJ',
  'tuku[acquire]V/t',
  'uŋ[people]N',
  'erim[enemy]N',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  'an[sky]N',
  'en[lord]N',
  'iri[city]N',
  'lah[bring]V/t',
  '_',
  'barsud[notation]N',
  '_',
  '_',
  '_',
  'šag[heart]N',
  'niŋsaga[goodness]N',
  '_',
  '_',
  'šeg[agree]V/i',
  'Ekur[1]TN',
  'teŋ[near]V/i',
  '_',
  '_',
  '_',
  '_',
  'ni[fear]N',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  'Subir[1]GN',
  '_',
  'duh[loosen]V/t',
  '_',
  '_',
  '_',
  '_',
  'tukul[weapon

# 2 PMI (Pointwise Mutual Information
[PMI](https://en.wikipedia.org/wiki/Pointwise_mutual_information) is a straightforward procedure for calculating the probability of two words appearing in the same window. This probability depends on

- the size of the vocabulary
- the size of the corpus
- the frequency of word A
- the frequency of word B

The probability of the collocation is compared to the actual number of collocates to come to a PMI score. 

A high PMI score indicates that the `collocation` of A and B is higher than what one would expect on the basis of pure chance. A PMI of 0 indicates that the collocation of A and B is exactly according to chance (independence). A negative PMI indicates that the two words actually repel each other.

In [10]:
count_lemmas = Counter()
count_lemma_pairs = Counter()
windowsize = widgets.IntSlider(min=2, max=25, value=7, description="window size")
windowsize

IntSlider(value=7, description='window size', max=25, min=2)

The following code collects all collocates, with window size defined in the slider above.

In [11]:
for text in tqdm(lemm_l):

    count_lemmas.update(text)

    for window in ngrams(text, windowsize.value):

        pairs = (
            tuple(sorted(pair))
            for pair in combinations(window, 2)
        )

        count_lemma_pairs.update(pairs)

  0%|          | 0/4811 [00:00<?, ?it/s]

Very common and very rare lemmas may be problematic in this computation. We remove words that appear less than 5 times. In the current code, no upper limit is set (that is, all frequent words are included).

In [12]:
print(f"{len(count_lemmas)} unique tokens before")
print(f"{sum(count_lemmas.values())} total tokens before")
min_count = 5
count_lemmas = Counter({
    token: count
    for token, count in count_lemmas.items()
    if count >= min_count and token != "_"
})

print(f"{len(count_lemmas)} unique tokens after")
print(f"{sum(count_lemmas.values())} total tokens after")
print('Most common:', count_lemmas.most_common(25))

4985 unique tokens before
355920 total tokens before
2393 unique tokens after
232614 total tokens after
Most common: [('ki[place]N', 3805), ('e[house]N', 3310), ('šu[hand]N', 3216), ('dug[speak]V/t', 2739), ('gal[big]V/i', 2625), ('lu[person]N', 2625), ('šag[heart]N', 2562), ('kur[mountain]N', 2338), ('me[be]V/i', 2151), ('ud[sun]N', 2073), ('ŋar[place]V/t', 2055), ('an[sky]N', 1934), ('kug[pure]V/i', 1890), ('e[leave]V/i', 1876), ('lugal[king]N', 1766), ('saŋ[head]N', 1723), ('igi[eye]N', 1646), ('dumu[child]N', 1627), ('ak[do]V/t', 1620), ('Enlil[1]DN', 1596), ('zid[right]V/i', 1514), ('gi[turn]V/i', 1504), ('gub[stand]V/i', 1468), ('a[water]N', 1431), ('mah[great]V/i', 1402)]


# Clean the Collocates
Remove all bigrams that contain a lemma that has been removed in the previous step.

In [13]:
count_lemma_pairs = Counter({
    (x, y): count
    for (x, y), count in count_lemma_pairs.items()
    if x in count_lemmas and y in count_lemmas
})

# Lookup Dictionaries
The following step creates two dictionaries. The dictionary `lemma2index` has the lemmas as keys and a counter as value. The dictionary `index2lemma` has those same counters as keys, and the lemmas as values. The counters will be used as indexes for the columns and rows of the matrix constructed below.

In [14]:
lemma2index, index2lemma = {}, {}
for i, x in enumerate(count_lemmas.keys()):
    lemma2index[x] = i
    index2lemma[i] = x

# Compute Lemma and Collocate Totals

In [15]:
lemma_total = sum(count_lemmas.values())
collocate_total = sum(count_lemma_pairs.values())

# Create PMI matrix

In the next cell the data is transformed into a (sparse) matrix, with rows and columns representing unique words, and the data in each cell representing the PMI score for the co-occurence of the two words. The variable `count_lemma_pairs` contains a list of all collocates in the format `('Unug[1]SN', 'šag[heart]N'): 1653`; meaning that the collocation of these two terms appears 1653 times. The dictionary `lemma2index` is used to translate each lemma into an index number and append the numbers to the lists `rows` and `cols`. The third element that the matrix function needs is `data`. Instead of simply entering the frequency in `data` we enter the PMI score. The formula for PMI is  $$log\frac{p(x,y)}{p(x)p(y)}$$ that is: the logarithm of the probability of encountering x *and* y, divided by the probability of x times the probability of y. The probabilities are computed by dividing the frequency of x and y by the total number of tokens and dividing the frequency of (x, y) by the total number of collocates. Negative PMI values are usually uninformative; PPMI (or Positive PMI) will set all negative results to 0.

One problem with PMI (or PPMI) is that it favors low frequency terms. If term A only appears 10 times in our dataset, each collocate with any other term will appear significant, because it occurs in 10% of all the occurences of term A. One solution is using PMI2, where the probability of the collocate is squared: PMI2 = $$log\frac{p(x,y)^2}{p(x)p(y)}$$

Still another possibility: PMIα: raise p(y) to the power of α, usually 0.75. PMIα = $$log\frac{p(x,y)}{p(x)p(y)^α}$$ 

In this notebook we use Positive PMI (PPMI).

In [16]:
pmi_flavor = "PPMI"

In [17]:
pmi_samples = {}
data, rows, cols = [], [], []
for (x, y), n in tqdm(count_lemma_pairs.items()):

    px = count_lemmas[x] / lemma_total
    py = count_lemmas[y] / lemma_total
    pxy = n / collocate_total

    score = max(log((pxy) / (px * py)), 0)

    i, j = lemma2index[x], lemma2index[y]
    if i != j:  # skip associating a word with itself
        rows += [i, j]
        cols += [j, i]
        data += [score, score]

        pmi_samples[(x, y)] = score

N = len(lemma2index)
PMI = csc_matrix(
    (data, (rows, cols)), shape=(N, N)
    )

mostcommon = sorted(
    pmi_samples.items(),
    key=lambda x: x[1],
    reverse=True
    )
print('%d non-zero elements' % PMI.count_nonzero())
print('Sample values\n', pformat(mostcommon[:10]))

  0%|          | 0/236039 [00:00<?, ?it/s]

403640 non-zero elements
Sample values
 [(('Endukuga[1]DN', 'Nindukuga[1]DN'), 9.330801152226739),
 (('Enmešara[1]DN', 'Ninzidanak[1]DN'), 9.289979157706483),
 (('harhar[instrument]N', 'sabitum[instrument]N'), 9.184618642048658),
 (('Endukuga[1]DN', 'Nindašurimak[1]DN'), 9.176650472399482),
 (('enkum[treasurer]N', 'nenkum[official]N'), 9.10765760091253),
 (('Endašurimak[1]DN', 'Nindašurimak[1]DN'), 9.09868893092977),
 (('Nindašurimak[1]DN', 'Nindukuga[1]DN'), 9.043119079774959),
 (('Endašurimak[1]DN', 'Endukuga[1]DN'), 9.022499792572223),
 (('Enul[1]DN', 'Ninul[1]DN'), 8.98124367605687),
 (('Zabu[1]GN', 'iš[mountain]N'), 8.958125866941566)]


# SVD
The sparse PMI matrix, which has as many rows and columns as there are unique lemmas, is factorized by using Single Value Decomposition. The number of columns is reduced by computing the first *k* factors that best explain the variance in the matrix. We end up with a vector for each word (a row in the new matrix) of length *k*.

SVD decomposes a matrix into three smaller matrices that, together, can reconstruct the original one. For our purposes, we only need the first of those. The underscores in the command line
```python
U, _, _ = svds(PMI, k)
```
represent the other two matrices, which are discarded.

In [18]:
k = 100
U, _, _ = svds(PMI, k)

# Normalizing
The resulting vectors (the rows in the reduced matrix) are normalized, so that each vector has length 1 (L2 normalization). The `preprocessing.normalize()`, part of the `sklearn` package, does that job. Normalization is necessary for computing cosine similarity in the next cell.

In [21]:
U = preprocessing.normalize(U, norm='l2')

# Save Vectors in Gensim Keyed Vectors Format
The `Gensim` package is used to create word vectors with `word2vec` (a neural network method that is suitable for very large corpora) and then compare and manipulate those vectors with a host of in-built functions. We have chosen another method for constructing the vectors (with PMI/SVD), but we can still use `Gensim` infrastructure to analyze our results. In order to do so we must save our vectors in a format that `Gensim` understands. This is called the Keyed Vectors format.

The Gensim Keyed Vectors format has a header that represents the dimensions of the data set (number of vectors and number of places in each vector). Each subsequent line begins with a key (the word or lemma), followed by the vector values, all separated by spaces. In order to do so, the matrix U is transformed into a Pandas DataFrame, and the lemmas are used as an index to that DataFrame. The Pandas method `to_csv` will now take care of creating the file.
When working in Windows, it is necessary to add the option `line_terminator='\n'`, or else there will be an additional line feed after each line.

In [22]:
vec = pd.DataFrame(U, index=lemma2index.keys())
vec = vec.loc[(vec != 0).any(axis=1)] # eliminate vectors that consist of zeros only.

with open("output/vec_file.txt", "w", encoding="utf8") as f:
    f.write(f"{len(vec)} {vec.shape[1]}\n")
    vec.to_csv(
        f,
        sep=" ",
        lineterminator="\n",
        header=False,
        float_format="%.6e",
    )

# Load Vectors in Gensim
The Gensim package allows one to compute word vectors with word2vec. Instead, we will load the vectors we computed with PMI/SVD above. 

For the code see https://stackoverflow.com/questions/27139908/load-precomputed-vectors-gensim

In [23]:
model_file = "output/vec_file.txt"
model = gensim.models.keyedvectors.Word2VecKeyedVectors.load_word2vec_format(model_file)

# Most Similar
We may now use the functions and methods available in Gensim, for instance the method `most_similar()`, which will find the vectors with the highest [cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity) to the target word(s).

The code in the function `mostsimilar()` uses the Gensim method `most_similar()` to display the top `n` words that are represented by vectors closests to the one of the target word. The target word is chosen from a dropdown menu and the number of similar words to be listed is defined by a slider.

The output is displayed in a `pandas` DataFrame.

In [24]:
def mostsimilar(change=None):
    lemma = dropdown.value
    topn = slider.value
    with out:
        clear_output(wait=True)
        data = model.most_similar(
            lemma, 
            topn=topn
        )
        df = pd.DataFrame(
            data, 
            columns = ["lemma", "sim"]
        )
        display(df)
    return 

# Custom Sort Order for the Dropdown Menu

The Gensim attribute `index_to_key` is a standard Python list containing all the words (keys) in the model's vocabulary, ordered by their frequency from most to least frequent. In order to use this list in a dropdown menu it needs to be reordered in alphabetical arrangement, with all the special characters used in Sumerian and Akkadian in their proper position. The variable `sortorder` is a string that contains all possible characters in order. This string is transformed into the dictionary `sort_index` where each character (the key of the dictionary) is associated with an index number (the value). Thus the letter `a` is associated with `5`, and `â` with `7`. This dictionary is used with the standard Python function `sorted()` to transform the `index_to_key` list into the alphabetically arranged case-insensitive list `word_options`. 

In [25]:
sortorder = (
    " []'ʾaāâbcdeēêfgŋhiīî"
    "jklmnopqrsṣštṭuūûvwxyz"
    "0123456789₀₁₂₃₄₅₆₇₈₉ₓ"
    "{}().-/~?!@×|&<>"
)
sort_index = {
    c: i for i, c in enumerate(sortorder)
}

# sort_index.get(c.casefold(), 9999) for c in w is a safety measure in case the
# vocabulary contains an unexpected character.
# use custom sort order

word_options = sorted(
    model.index_to_key, 
    key=lambda w: [
        sort_index.get(c.casefold(), 9999)
        for c in w
    ]
)

# User Interface
The code in this block creates and displays the dropdown menu that lists all the words in the vocabulary, and a slider that defines how many of the most similar words should be displayed. When the user changes anything either in the dropdown or in the slider, the `observe()` method of the widget calls the `mostsimilar()` function defined above, with the result that new output is produced.

The `Output()` widget creates a space where the out put can be displayed. The `VBox` (Vertical Box) aligns dropdown menu, slider, and output vertically.

In [26]:
# define the widgets

dropdown = widgets.Dropdown(
    options = word_options, 
    description = 'Target Word'
)
slider = widgets.IntSlider(
    value=5, 
    min=1, 
    max = 25, 
    description = 'Top N')

out = widgets.Output(
    layout=widgets.Layout(
        width="600px"
    )
)

# observe changes in the widgets

dropdown.observe(
    mostsimilar, 
    names="value")
slider.observe(
    mostsimilar, 
    names="value"
)

# arrange the widgets and display
ui = widgets.VBox([
    dropdown,
    slider,
    out
])

display(ui)

# initial run of the `mostsimilar()` function
mostsimilar()

# Plotting words for way/street/route

We will take four or five words that are all partly synonymous and display the ten most similar words of each of these. This may illustrate the differences between those words by showing how they elicit different kinds of associations. The result will be displayed in a graph that visualizes the (semantic) distances between the words.

# Word Categories
The function `word_categories()` takes one or more (up to five) words and finds for each of them them the top `n` most similar words. The target word and the `n` most similar words form a semantic group, that in a following cell will be projected onto a two-dimensional space, illustrating the distance between those words. By choosing multiple words you can see how these semantic groups (ordered around the target word) are located in the vector space in relation to each other.

The output of this function is a dictionary (`word_d`) where each of the words collected is associated with a number that represents the word category. Thus, if the input were `lugal[king]N`, `udu[sheep]N` with topn=5, we get five words that are most similar to `lugal` and five most similar to `udu`; those in the `lugal` list receive the number 0, those in the `udu` list the number 1. The original words (`lugal` and `udu` in this example) are also added. A special category is created for words that end up in more than one "most similar" list.

In [27]:
def word_categories(model, words, topn=15):
    """word_categories takes an iterable with one or more words from the vocabulary of the model.
    For each word a dictionary of similar words (including the target word) is construed.
    The value of each word is numerical (integer) and indicates the category to which it belongs.
    The function returns a dictionary."""
    if len(words) > 5: 
        words = words[:5]
    word_d = {}
    for idx, word in enumerate(words):
        w = model.most_similar(word, topn=topn)
        w = [m[0] for m in w] # keep only the words, not the cosine similarity
        w.append(word) # add the target word
        for item in w:
            if item in word_d:
                word_d[item] = f"{word_d[item]}; {word}" #  words that appear in
                                        # more than one `most similar` list.
            else:
                word_d[item] = word
    return word_d

# Dictionary to link word to stable identifier (oid) 
The directory `output` contains a dictionary that links each lemma to the stable identifier (oid) of that word in [ePSD2](http://oracc.org/epsd2). This dictionary is used to create live links in the visualization.

In [28]:
with open("output/lemma2oid.p", "rb") as r:
    lemma2oid = pickle.load(r)

# Project and Visualize Related Words
Pick a maximum of 5 words. For each of those words the 10 most similar words are selected. The words are projected on a two-dimensional space with t-SNE, a visualization is created with bokeh.

In [29]:
DEFAULT_COLORS = {
    0 : "#4E79A7",  # blue
    1 : "#F28E2B",  # orange
    2 : "#E15759",  # red
    3 : "#76B7B2",  # teal
    4 : "#59A14F",  # green
    5 : "#EDC948",  # yellow
    6 : "#B07AA1",  # purple
    7 : "#FF9DA7",  # pink
    8 : "#9C755F",  # brown
    9 : "#BAB0AC",  # gray
}

def tsne_bokeh2(
    model,
    words,
    fontsize="12pt",
    color_map=None,
    width=800,
    height=600,
    random_state=23,
):
    """
    Create a 2D t-SNE visualization using Bokeh.

    Parameters
    ----------
    model : dict-like
        Word embedding model.
    words : list[str]
        Seed/category words.
    fontsize : str
        Label font size.
    color_map : dict[int, str] | None
        Optional category-to-color mapping.
    width : int
        Plot width.
    height : int
        Plot height.
    random_state : int
        Random seed for reproducibility.
    """

    if color_map is None:
        color_map = DEFAULT_COLORS

    topn = topnwords.value
    words_d = word_categories(model, words, topn)
 
    colors = {}
    for idx, category in enumerate(set(words_d.values())): 
        colors[category] = idx

    rows = []   

    for word, category in words_d.items():

        if word not in model:
            continue
       
        rows.append({
            "label": word,
            "vector": model[word],
            "legend": category,
            "oid": lemma2oid.get(word, ""),
            "color": color_map.get(colors[category], "gray"),
        })

    if len(rows) < 2:
        raise ValueError(
            "Need at least 2 valid words.")
    
    vectors = np.array([
        row["vector"]
        for row in rows
    ])        
            
    # t-SNE requires perplexity < n_samples
    perplexity = max(
        2,
        min(30, len(vectors) // 3)
    )

    tsne = TSNE(
        perplexity=perplexity,
        n_components=2,
        init="pca",
        max_iter=2500, 
        random_state=random_state,
    )
    
    embedding = tsne.fit_transform(vectors)

    source = ColumnDataSource(
        data=dict(
            x=embedding[:, 0],
            y=embedding[:, 1],
            labels=[r["label"] for r in rows],
            colors=[r["color"] for r in rows],
            legend=[r["legend"] for r in rows],
            oid=[r["oid"] for r in rows],
        )
    )

    plot = figure(
        width=width,
        height=height,
        tools=[
            "pan",
            "wheel_zoom",
            "zoom_in",
            "zoom_out",
            "tap",
            "reset",
            "box_zoom",
            "save",
        ],
        toolbar_location="right",
    )

    plot.scatter(
        x="x",
        y="y",
        size=8,
        fill_color="colors",
        line_alpha=0,
        source=source,
        legend_group="legend",
    )
    
    labels = LabelSet(
        x="x",
        y="y",
        text="labels",
        source=source,
        x_offset=3,
        y_offset=3,
        text_font_size=fontsize,
    )

    plot.add_layout(
        Title(
            text=(f"Vector length = {str(model.vector_size)}; "
            f"Topn = {str(topnwords.value)}; "
            f"PMI Window Size = {str(windowsize.value)}; "
            f"PMI flavor = {pmi_flavor}; "
            f"Projects = {projects}.")
        ),
        "above"
    )
    plot.add_layout(labels)

    #plot.legend.location = "bottom_right"
    plot.legend.orientation = "horizontal"
    plot.add_layout(plot.legend[0], "below")
    
    plot.add_tools(HoverTool(
        tooltips=[
            ("Word", "@labels"),
            ("Category", "@legend"),
        ]
    ))

    taptool = plot.select_one(TapTool)

    if taptool:
        taptool.callback = OpenURL(
            url="http://oracc.org/epsd2/@oid"
        )

    return plot

# Create the controls
The following code creates interactive controls (called `Widgets`) to select target words, control the number of similar words to be represented, etc.

In [30]:
# -----------------------------
# controls
# -----------------------------

word_selector = widgets.SelectMultiple(
    options=word_options,
    description="Words",
    value=("inti[way]N", "kaskal[way]N", "harran[route]N", "esir[street]N"),
)
topnwords = widgets.IntSlider(min=2, max=20, value=15, description="top n words")

# -----------------------------
# dedicated plot area
# -----------------------------

plot_output = widgets.Output()
current_plot = None
# -----------------------------
# update callback
# -----------------------------

def update(change=None):

    global current_plot
    with plot_output:

        plot_output.clear_output(wait=True)

        current_plot = tsne_bokeh2(
            model=model,
            words=word_selector.value,
            fontsize="12pt",
            color_map=None,
            width=900,
            height=600,
            random_state=23,
        )

        show(current_plot)

# -----------------------------
# observer
# -----------------------------

word_selector.observe(update, names="value")
topnwords.observe(update, names="value")

# -----------------------------
# layout
# -----------------------------

ui = widgets.HBox(
    [word_selector,
    topnwords
])

out = widgets.VBox([
    ui,
    plot_output
])

display(out)

# initial render
update()

Save as `html` file for further inspection.

In [31]:
output_file(
    "output/tsne_plot.html",
    title="Sumerian Semantic Space"
)
save(current_plot, resources=INLINE);